In [ ]:
!pip -q install -U ultralytics
from ultralytics import YOLO
import os, yaml, json, inspect
import torch
from google.colab import drive
drive.mount('/content/drive')
print('Torch:', torch.__version__)

In [ ]:
def add_module_classes(mod):
    try:
        safe = []
        for name, obj in vars(mod).items():
            try:
                if inspect.isclass(obj):
                    safe.append(obj)
            except Exception:
                pass
        if safe:
            torch.serialization.add_safe_globals(safe)
    except Exception:
        pass

import torch.nn as nn
add_module_classes(nn)
add_module_classes(nn.modules)
from torch.nn import modules as nnmods
for sub in (
    getattr(nnmods, 'container', None), getattr(nnmods, 'conv', None), getattr(nnmods, 'batchnorm', None),
    getattr(nnmods, 'activation', None), getattr(nnmods, 'pooling', None), getattr(nnmods, 'linear', None),
    getattr(nnmods, 'upsampling', None), getattr(nnmods, 'dropout', None), getattr(nnmods, 'normalization', None),
    getattr(nnmods, 'sparse', None)):
    if sub is not None:
        add_module_classes(sub)

try:
    from torch.nn.modules.container import Sequential, ModuleList, ModuleDict
    torch.serialization.add_safe_globals([Sequential, ModuleList, ModuleDict])
except Exception:
    pass

try:
    import ultralytics
    add_module_classes(ultralytics)
    from ultralytics import nn as ulnn
    add_module_classes(ulnn)
    from ultralytics.nn import tasks, modules as ulnnmods
    add_module_classes(tasks)
    add_module_classes(ulnnmods)
    from ultralytics.nn.tasks import DetectionModel, SegmentationModel, PoseModel, ClassificationModel
    torch.serialization.add_safe_globals([DetectionModel, SegmentationModel, PoseModel, ClassificationModel])
except Exception as e:
    print('Allow-list note:', e)

print('Safe globals allow-list applied.')

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/leaf_det'    
OUT_DIR   = '/content/drive/MyDrive/leaf_detector_runs'
os.makedirs(OUT_DIR, exist_ok=True)
data_yaml_path = os.path.join(DATA_ROOT, 'data.yaml')
print('DATA_ROOT exists:', os.path.isdir(DATA_ROOT))
print('data.yaml exists:', os.path.isfile(data_yaml_path))
if os.path.isfile(data_yaml_path):
    print(open(data_yaml_path).read())
else:
    data_yaml = {'path': DATA_ROOT, 'train': 'images/train', 'val': 'images/val', 'names': ['leaf']}
    with open(data_yaml_path, 'w') as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)
    print('Created minimal data.yaml (single class: leaf)')

In [ ]:
MODEL  = 'yolov8n.pt'   
EPOCHS = 20
IMGSZ  = 640
BATCH  = 16

model = YOLO(MODEL)
results = model.train(
    data=data_yaml_path,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=OUT_DIR,
    name='leafdet',
    optimizer='AdamW',
    cos_lr=True,
    lr0=0.001,
)
best_pt = os.path.join(OUT_DIR, 'leafdet', 'weights', 'best.pt')
print('Best weights:', best_pt)

In [ ]:
INPUT_SIZE = 480   
model = YOLO(best_pt)
onnx_path = model.export(format='onnx', imgsz=INPUT_SIZE, opset=12, dynamic=True)
print('Exported ONNX:', onnx_path)

import shutil
final_onnx = os.path.join(OUT_DIR, f'leafdet_yolov8n_{INPUT_SIZE}.onnx')
shutil.copy2(onnx_path, final_onnx)
print('Saved:', final_onnx)

names = None
try:
    names = model.model.names if hasattr(model, 'model') else None
except Exception:
    names = None
if names is None:
    try:
        with open(os.path.join(DATA_ROOT, 'data.yaml'), 'r') as f:
            y = yaml.safe_load(f)
        names = y.get('names')
    except Exception:
        names = None
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys(), key=lambda k: int(k))]
classes_json = os.path.join(OUT_DIR, 'leafdet_classes.json')
with open(classes_json, 'w') as f:
    json.dump(names, f, indent=2)
print('Saved class names to:', classes_json)
print('Class names:', names)

In [ ]:
from glob import glob
sample_imgs = glob(os.path.join(DATA_ROOT, 'images', 'val', '*'))[:4]
if sample_imgs:
    preds = YOLO(best_pt).predict(source=sample_imgs, conf=0.25, imgsz=INPUT_SIZE)
    preds
else:
    print('No sample images found in val. Skipping preview.')